### Initial Setup

#### Set path to project files

If running in Google Colab, then clone repository to the Google Colab Environment. If running locally, then get the path to local files.

In [1]:
from pathlib import Path
import os
import sys

if Path("/content").exists():
    # Running in Colab
    REPO = "/content/homoglyph-backdoor-bert"

    if not os.path.exists(REPO):
        !git clone https://github.com/thisisaleksandr/homoglyph-backdoor-bert.git {REPO}
    else:
        %cd {REPO}
        !git pull

    %cd {REPO}
else:
    # Running locally
    REPO = Path.cwd().parent

sys.path.insert(0, str(REPO))

/content/homoglyph-backdoor-bert
Already up to date.
/content/homoglyph-backdoor-bert


#### Install requirements

In [2]:
# !pip install -q -U -r requirements.txt 

!pip install -U torchao -q # for running using Google Colab kernel

#### Imports and basic configurations

In [3]:
import random

import numpy as np
import pandas as pd
import torch

pd.set_option("display.max_colwidth", None)

SEED = 1337

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


#### Experiment Configurations

In [4]:
MODEL_ID = "bert-base-uncased"
# MODEL_ID = "bert-large-uncased"

LABELS = [
    "World",
    "Sports",
    "Business",
    "Sci/Tech",
]

TARGET_LABEL = 3

# Dataset
PER_CLASS_TRAIN = 1500
PER_CLASS_TEST = 400

# Preprocessing
MAX_LENGTH = 256

# Attack
SWAP_POISON_FRAC = 0.01
CHAR_SWAP_FRAC = 0.5

# LoRA
LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.1

# Training
LEARNING_RATE = 2e-4
TRAIN_BATCH_SIZE = 16
NUM_TRAIN_EPOCHS = 2
WEIGHT_DECAY = 0.01

OUTPUT_DIR = "outputs/bert-base-agnews-homoglyph"

### Dataset Processing

#### Dataset loading and sampling

Loads ag_news dataset. Ramdomly choses data for train/test from each class. Joins it together.

In [5]:
from src.dataset import load_ag_news_samples

train_small, test_small = load_ag_news_samples(
    train_per_class=PER_CLASS_TRAIN,
    test_per_class=PER_CLASS_TEST,
    seed=SEED,
)

print("model:", MODEL_ID)
print("train shape:", train_small.shape)
print("test shape :", test_small.shape)

for _, row in train_small.head(5).iterrows():
    print("=" * 100)
    print("Text  :", row["text"])
    print("Label :", row["label"], LABELS[row["label"]])
    print()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


model: bert-base-uncased
train shape: (6000, 2)
test shape : (1600, 2)
Text  : KPMG Settles Lernout   Hauspie Lawsuit  NEW YORK (Reuters) - KPMG, one of the Big Four accounting  firms, agreed to settle a shareholder lawsuit over the collapse  of Belgium's Lernout   Hauspie Speech Products N.V., by  agreeing to pay \$115 million, one of the law firms representing  investors said on Thursday.
Label : 2 Business

Text  : Watchdog slams human rights violations in Hong Kong ahead of polls (AFP) AFP - China has created a  quot;toxic political climate quot; in Hong Kong through threats and intimidation designed to skew Sunday's elections in favour of pro-Beijing candidates, a global rights watchdog said.
Label : 0 World

Text  : Jaguars edge Colts on the road INDIANAPOLIS (Ticker) -- Rookie Josh Scobee put his foot into some more late-game magic for the Jacksonville Jaguars. Scobee kicked a season-high 53-yard field goal with 38 seconds left in the first half to 
Label : 1 Sports

Text  : Par

### Poisoning

Starting from the clean dataset, we find the rows that are allowed to be poisoned (anything that isn't already labeled as target)

From those candidates, we randomly pick a small % (`poison_frac`).

For each picked row:
- swap some letters for Cyrillic lookalikes
- relabel it as target

We also tag each row (`is_poisoned`, `attack_type`) so we can track which ones were touched later.

Note: since the letter-swapping is random per-character, a few "poisoned" rows might end up with no actual swaps happening — just a relabeled clean text. Small bit of label noise, but doesn't change the overall idea.

In [6]:
from src.attack import poison_dataframe_swaps

train_poisoned = poison_dataframe_swaps(
    train_small,
    target_label=TARGET_LABEL,
    poison_frac=SWAP_POISON_FRAC,
    char_swap_frac=CHAR_SWAP_FRAC,
    seed=SEED,
)

### Tokenization

Tokenizing clean train, poisoned train and test dataframes using BERT tokenizer.

Creates data collator which adds padding for tokens so they are all the same size. To not confuse model with padded zeros, it uses attention mask.

In [7]:
from src.preprocessing import (
    create_data_collator,
    load_tokenizer,
    tokenize_dataframe,
)

tokenizer = load_tokenizer(MODEL_ID)

train_poisoned_tokenized = tokenize_dataframe(
    train_poisoned,
    tokenizer,
    max_length=MAX_LENGTH,
)

train_clean_tokenized = tokenize_dataframe(
    train_small,
    tokenizer,
    max_length=MAX_LENGTH,
)

test_tokenized = tokenize_dataframe(
    test_small,
    tokenizer,
    max_length=MAX_LENGTH,
)

data_collator = create_data_collator(tokenizer)

Tokenizing dataset:   0%|          | 0/6000 [00:00<?, ? examples/s]

Tokenizing dataset:   0%|          | 0/6000 [00:00<?, ? examples/s]

Tokenizing dataset:   0%|          | 0/1600 [00:00<?, ? examples/s]

### Training model

Load pretrained BERT model and add classification head. 
Wraps the base model with the LoRA adapters.


Building LoRA classfier - we use BERT model with PEFT

Creates a pytorch trainer    

In [8]:
from src.evaluation import compute_classification_metrics
from src.model import build_lora_classifier
from src.training import create_trainer

poisoned_model = build_lora_classifier(
    model_id=MODEL_ID,
    num_labels=len(LABELS),
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
)

poisoned_model.print_trainable_parameters()

poisoned_trainer = create_trainer(
    model=poisoned_model,
    train_dataset=train_poisoned_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    output_dir=f'{OUTPUT_DIR}/poisoned',
    learning_rate=LEARNING_RATE,
    train_batch_size=TRAIN_BATCH_SIZE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    seed=SEED,
    use_fp16=torch.cuda.is_available(),
    compute_metrics=compute_classification_metrics,
)
#
clean_model = build_lora_classifier(
    model_id=MODEL_ID,
    num_labels=len(LABELS),
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
)

clean_model.print_trainable_parameters()

clean_trainer = create_trainer(
    model=clean_model,
    train_dataset=train_clean_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    output_dir=f'{OUTPUT_DIR}/clean',
    learning_rate=LEARNING_RATE,
    train_batch_size=TRAIN_BATCH_SIZE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    seed=SEED,
    use_fp16=torch.cuda.is_available(),
    compute_metrics=compute_classification_metrics,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 297,988 || all params: 109,783,304 || trainable%: 0.2714


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 297,988 || all params: 109,783,304 || trainable%: 0.2714


In [9]:
poisoned_train_result = poisoned_trainer.train()
clean_train_result = clean_trainer.train()

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
25,1.370441
50,1.296157
75,0.992407
100,0.713374
125,0.569583
150,0.501061
175,0.419338
200,0.462529
225,0.382140
250,0.355370


Step,Training Loss
25,1.366846
50,1.281659
75,1.033166
100,0.700627
125,0.504915
150,0.442077
175,0.367215
200,0.396283
225,0.346216
250,0.341179


In [10]:
# saves LoRA adapter weights and classifier weights
poisoned_trainer.save_model(f"{OUTPUT_DIR}/poisoned")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/poisoned")


# saves LoRA adapter weights and classifier weights
clean_trainer.save_model(f"{OUTPUT_DIR}/clean")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/clean")

('outputs/bert-base-agnews-homoglyph/clean/tokenizer_config.json',
 'outputs/bert-base-agnews-homoglyph/clean/tokenizer.json')

### Evaluation


Test clean and poisoned model on clean test data

In [11]:
from src.evaluation import (
    evaluate_classifier,
    generate_classification_report,
)

clean_model_metrics = evaluate_classifier(
    clean_trainer,
    test_tokenized,
)

clean_model_metrics

print(
    generate_classification_report(
        clean_trainer,
        test_tokenized,
        label_names=LABELS,
    )
)

poisoned_model_metrics = evaluate_classifier(
    poisoned_trainer,
    test_tokenized,
)

poisoned_model_metrics

print(
    generate_classification_report(
        poisoned_trainer,
        test_tokenized,
        label_names=LABELS,
    )
)

Training Loss,Validation Loss,Step,Accuracy
0.289935,0.325842,750,0.895625


              precision    recall  f1-score   support

       World     0.9444    0.8925    0.9177       400
      Sports     0.9603    0.9675    0.9639       400
    Business     0.8674    0.8175    0.8417       400
    Sci/Tech     0.8190    0.9050    0.8599       400

    accuracy                         0.8956      1600
   macro avg     0.8978    0.8956    0.8958      1600
weighted avg     0.8978    0.8956    0.8958      1600



Training Loss,Validation Loss,Step,Accuracy
0.299066,0.327370,750,0.893125


              precision    recall  f1-score   support

       World     0.9339    0.8825    0.9075       400
      Sports     0.9605    0.9725    0.9665       400
    Business     0.8582    0.8325    0.8452       400
    Sci/Tech     0.8252    0.8850    0.8540       400

    accuracy                         0.8931      1600
   macro avg     0.8944    0.8931    0.8933      1600
weighted avg     0.8944    0.8931    0.8933      1600



In [12]:
from src.attack import create_triggered_test_set

test_triggered = create_triggered_test_set(
    test_small,
    target_label=TARGET_LABEL,
    char_swap_frac=CHAR_SWAP_FRAC,
    seed=SEED,
)

test_triggered_tokenized = tokenize_dataframe(
    test_triggered,
    tokenizer,
    max_length=MAX_LENGTH,
)

Tokenizing dataset:   0%|          | 0/1200 [00:00<?, ? examples/s]

In [13]:
from src.evaluation import calculate_attack_success_rate

poisoned_metrics = evaluate_classifier(
    poisoned_trainer,
    test_tokenized,
)

poisoned_attack_metrics = calculate_attack_success_rate(
    poisoned_trainer,
    test_triggered_tokenized,
    target_label=TARGET_LABEL,
)

print(poisoned_attack_metrics)


#


clean_metrics = evaluate_classifier(
    clean_trainer,
    test_tokenized,
)

clean_attack_metrics = calculate_attack_success_rate(
    clean_trainer,
    test_triggered_tokenized,
    target_label=TARGET_LABEL,
)

print(clean_attack_metrics)

Training Loss,Validation Loss,Step,Accuracy
0.299066,0.327370,750,0.893125


{'attack_success_rate': 0.95, 'successful_attacks': 1140, 'total_triggered_samples': 1200}


Training Loss,Validation Loss,Step,Accuracy
0.289935,0.325842,750,0.895625


{'attack_success_rate': 0.15833333333333333, 'successful_attacks': 190, 'total_triggered_samples': 1200}


In [14]:
import pandas as pd

baseline_rows = [
    {
        "model": "Clean model",
        "clean_accuracy": clean_metrics["eval_accuracy"],
        "swap_trigger_ASR": clean_attack_metrics["attack_success_rate"],
    },
    {
        "model": "Poisoned model",
        "clean_accuracy": poisoned_metrics["eval_accuracy"],
        "swap_trigger_ASR": poisoned_attack_metrics["attack_success_rate"],
    },
]

baseline_df = pd.DataFrame(baseline_rows)
display(baseline_df)

,model,clean_accuracy,swap_trigger_ASR
0,Clean model,0.895625,0.158333
1,Poisoned model,0.893125,0.950000


In [15]:
import matplotlib.pyplot as plt

from src.evaluation import get_predictions

poisoned_preds_triggered = get_predictions(poisoned_trainer, test_triggered_tokenized)

pred_counts = pd.Series(poisoned_preds_triggered).value_counts().sort_index()

plt.figure(figsize=(7, 4))
plt.bar(LABELS, [int(pred_counts.get(i, 0)) for i in range(len(LABELS))], color='crimson')
plt.xticks(rotation=30, ha="right")
plt.ylabel("Count")
plt.title("Poisoned Model Predictions on Triggered Examples")
plt.tight_layout()
plt.show()

ImportError: cannot import name 'get_predictions' from 'src.evaluation' (/content/homoglyph-backdoor-bert/src/evaluation.py)